# Sprint 3 — Representation & Clustering (Colab launcher)

This notebook only **launches** `sprint3_colab_experiment.py` and **reads** its outputs. The script is the source of
truth; no experiment logic lives here and nothing is recalculated in this notebook.

Controlled 2 × 2 experiment on the frozen input `sprint3_task_input.csv` (3,015 task statements):

1. **E5 + K-Means** (`intfloat/e5-base-v2`)
2. **E5 + HDBSCAN**
3. **Sentence-T5 + K-Means** (`sentence-transformers/sentence-t5-base`)
4. **Sentence-T5 + HDBSCAN**

**Before running:** upload `Sprint3_Experiment_Colab.ipynb`, `sprint3_colab_experiment.py` and `sprint3_task_input.csv`
to **My Drive/CAPESTONE PROJECT/**, then select *Runtime → Change runtime type → GPU*. Run the cells top to bottom.
If the runtime disconnects, re-run cells 3–6: the script resumes from its checkpoints.

In [1]:
# CELL 2 — check GPU
!nvidia-smi
import torch
print("torch.cuda.is_available():", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU - the experiment still runs on CPU, only slower (Runtime > Change runtime type > GPU).")

Thu Oct 22 21:07:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# CELL 3 — mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# CELL 4 — paths (verified before continuing)
import os

PROJECT_DIR = "/content/drive/MyDrive/CAPESTONE PROJECT"
SCRIPT = f"{PROJECT_DIR}/sprint3_colab_experiment.py"
INPUT = f"{PROJECT_DIR}/sprint3_task_input.csv"
OUTPUT = f"{PROJECT_DIR}/sprint3_results"
ZIP = f"{PROJECT_DIR}/Sprint3_Experiment_Results.zip"

for name, path in [("SCRIPT", SCRIPT), ("INPUT", INPUT), ("OUTPUT", OUTPUT), ("ZIP", ZIP)]:
    print(f"{name:7s} {path}")

missing = [p for p in (SCRIPT, INPUT) if not os.path.isfile(p)]
if missing:
    raise FileNotFoundError("Upload these files to My Drive/CAPESTONE PROJECT/ first: " + ", ".join(missing))
print("\nSCRIPT and INPUT found.")

SCRIPT  /content/drive/MyDrive/CAPESTONE PROJECT/sprint3_colab_experiment.py
INPUT   /content/drive/MyDrive/CAPESTONE PROJECT/sprint3_task_input.csv
OUTPUT  /content/drive/MyDrive/CAPESTONE PROJECT/sprint3_results
ZIP     /content/drive/MyDrive/CAPESTONE PROJECT/Sprint3_Experiment_Results.zip

SCRIPT and INPUT found.


In [4]:
# CELL 5 — install dependencies (exactly the packages sprint3_colab_experiment.py imports;
# packages Colab already provides are left as they are)
!pip install -q numpy pandas scipy scikit-learn matplotlib torch transformers sentence-transformers sentencepiece hdbscan umap-learn

In [5]:
# CELL 6 — run the complete experiment
!python "/content/drive/MyDrive/CAPESTONE PROJECT/sprint3_colab_experiment.py" \
  --input "/content/drive/MyDrive/CAPESTONE PROJECT/sprint3_task_input.csv" \
  --output "/content/drive/MyDrive/CAPESTONE PROJECT/sprint3_results"

[21:08:44 +    24s] device=cuda gpu=Tesla T4 torch=2.11.0+cu130 sklearn=1.6.1 hdbscan_impl=hdbscan
[21:08:46 +    25s] input rows=3015 postings=188 providers={'gemini': 2719, 'groq': 296} sha256=010795fed2864f99... frozen=True
[21:08:46 +    25s] cross-provider paired wordings available (non-identical): 432
[21:08:46 +    25s] === representation e5: intfloat/e5-base-v2 ===
[21:08:46 +    25s] e5: encoding 3015 statements with intfloat/e5-base-v2 on cuda
config.json: 100% 650/650 [00:00<00:00, 2.81MB/s]
tokenizer_config.json: 100% 314/314 [00:00<00:00, 1.88MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 82.1MB/s]
tokenizer.json: 100% 711k/711k [00:00<00:00, 139MB/s]
special_tokens_map.json: 100% 125/125 [00:00<00:00, 753kB/s]

model.safetensors: downloading bytes:   0% 0.00/438M [00:00<?, ?B/s]
model.safetensors: downloading bytes:   5% 19.8M/438M [00:01<00:23, 18.1MB/s, 41.8kB/s  ]
model.safetensors: downloading bytes:  40% 173M/438M [00:02<00:01, 157MB/s, 15.0MB/s  ]
model.safetensors: 

In [6]:
# CELL 7 — verify outputs (read-only)
import json, os
from pathlib import Path
from IPython.display import Markdown, display

if os.path.isfile(ZIP):
    print(f"ZIP: {ZIP}\nsize: {os.path.getsize(ZIP) / 1e6:.1f} MB")
else:
    print(f"ZIP NOT FOUND: {ZIP} - check the output of cell 6 for errors.")

out = Path(OUTPUT)
print("\nOutput directories:")
for d in ["embeddings", "evaluation", "final", "figures", "input"]:
    p = out / d
    n = len([f for f in p.rglob("*") if f.is_file()]) if p.exists() else 0
    print(f"  {d:11s} {'OK ' if p.exists() else 'MISSING'} {n} files")

manifest = None
mf = out / "run_manifest.json"
if mf.exists():
    manifest = json.loads(mf.read_text())
    ch = manifest.get("chosen_configuration", {})
    print("\nWinning pipeline (from run_manifest.json):", ch.get("config_id"),
          "|", ch.get("embedding_model"), "|", ch.get("clustering_method"), "|", ch.get("parameters"))

ms = out / "evaluation" / "model_selection.md"
if ms.exists():
    display(Markdown(ms.read_text()))
else:
    print("model_selection.md not found")

ZIP: /content/drive/MyDrive/CAPESTONE PROJECT/Sprint3_Experiment_Results.zip
size: 20.3 MB

Output directories:
  embeddings  OK  10 files
  evaluation  OK  13 files
  final       OK  3 files
  figures     OK  8 files
  input       OK  1 files

Winning pipeline (from run_manifest.json): EXP_3_T5_KMEANS__k=100 | sentence-transformers/sentence-t5-base | kmeans | k=100


# Sprint 3 Model Selection

Decision procedure (pre-declared in the script): (1) reject pathological configurations ({"min_stability_ari": 0.5, "max_dominant_share": 0.2, "max_tiny_cluster_share": 0.25, "max_noise_share": 0.5, "min_clusters": 10, "max_clusters": 300, "min_cohesion_lift": 0.05}); (2) lexicographic hierarchy with tolerances semantic_cohesion_lift (tol 0.02), stability_ari (tol 0.05), granularity_score (tol 0.0), robustness (tol 0.05), silhouette (tol 0.01). A level only breaks ties among configurations that are within tolerance of the best on all higher levels. No single metric decides.

Cosine-based cohesion is compared across embedding models only as a chance-adjusted **lift** ((cohesion - random-partition cohesion) / (1 - random-partition cohesion)) because raw cosine scales differ between E5 and Sentence-T5. Silhouette is measured in each model's own PCA space and is the lowest-priority criterion.

## 1. E5 vs Sentence-T5

**Verdict: INCONCLUSIVE - the evidence does not clearly favour either representation**

- semantic_cohesion_lift: E5 mean 0.156 vs T5 0.151; E5 higher in 3/10 matched configurations -> tie
- stability_ari: E5 mean 0.495 vs T5 0.465; E5 higher in 10/13 matched configurations -> e5
- provider_pair_kappa: E5 mean -2.984 vs T5 -3.008; E5 higher in 4/10 matched configurations -> tie
- paraphrase MRR E5 0.895 vs T5 0.891 (diff 95% CI [-0.015, 0.022]) -> tie

| criterion | matched configs | E5 | T5 | E5 better in |
|---|---:|---:|---:|---:|
| semantic_cohesion_lift | 10.0 | 0.156 | 0.151 | 3.0 |
| stability_ari | 13.0 | 0.495 | 0.465 | 10.0 |
| provider_pair_kappa | 10.0 | -2.984 | -3.008 | 4.0 |
| silhouette | 10.0 | 0.078 | 0.072 | 6.0 |
| low_confidence_rate | 13.0 | 0.161 | 0.166 | 4.0 |
| paraphrase_mrr | nan | 0.895 | 0.891 | nan |

Top-5 neighbour sets agree between E5 and T5 with mean Jaccard 0.441. Inspect `nearest_neighbor_examples.csv` before accepting the verdict.

## 2. K-Means vs HDBSCAN (within Sentence-T5)

Candidates: K-Means `EXP_3_T5_KMEANS__k=100` vs HDBSCAN `EXP_4_T5_HDBSCAN__min_cluster_size=20_min_samples=10` -> **kmeans** (decided by semantic_cohesion_lift).

- no non-pathological configuration - choosing among the least-flagged
- semantic_cohesion_lift: best=0.290 tol=0.02 -> kept EXP_3_T5_KMEANS__k=100

| config_id | n_clusters | noise_share | semantic_cohesion_lift | stability_ari | granularity_score | robustness | silhouette | low_confidence_rate | pathology_flags |
|---|---|---|---|---|---|---|---|---|---|
| EXP_3_T5_KMEANS__k=100 | 100 | 0.000 | 0.290 | 0.385 | 1.000 | 0.464 | 0.069 | 0.356 | unstable |
| EXP_4_T5_HDBSCAN__min_cluster_size=20_min_samples=10 | 3 | 0.958 | 0.189 | 0.620 | 0.000 | -0.069 | 0.171 | 0.000 | too_few_clusters;dominant_cluster;excessive_noise |

## 3. Winning pipeline

- configuration: `EXP_3_T5_KMEANS__k=100`
- embedding model: sentence-transformers/sentence-t5-base
- clustering method: kmeans
- exact parameters: k=100 (PCA 50 components, cumulative variance 0.607; seed 42)
- leaf topic count: 100
- noise / unassigned (before post-hoc rule): 0 (0.0%)
- semantic cohesion: 0.914 (chance 0.879, lift 0.290)
- stability: subsample ARI 0.385 ± 0.017; seed-restart ARI 0.418
- robustness: provider-pair kappa 0.684, cross-representation ARI 0.245, contamination stress-test major-topic preservation 31.1% (ARI 0.358)
- decided by: robustness

  - no non-pathological configuration - choosing among the least-flagged
  - semantic_cohesion_lift: best=0.290 tol=0.02 -> kept EXP_1_E5_KMEANS__k=100, EXP_3_T5_KMEANS__k=100
  - stability_ari: best=0.404 tol=0.05 -> kept EXP_1_E5_KMEANS__k=100, EXP_3_T5_KMEANS__k=100
  - granularity_score: best=1.000 tol=0.0 -> kept EXP_1_E5_KMEANS__k=100, EXP_3_T5_KMEANS__k=100
  - robustness: best=0.464 tol=0.05 -> kept EXP_3_T5_KMEANS__k=100

## 4. Second-best pipeline

`EXP_1_E5_KMEANS__k=100` - intfloat/e5-base-v2, kmeans, k=100; 100 topics, noise 0.0%, cohesion lift 0.280, stability ARI 0.404.

## 5. Why the winner beat the second-best

| criterion | tolerance | winner | second | difference |
|---|---:|---:|---:|---:|
| semantic_cohesion_lift | 0.02 | 0.290 | 0.280 | 0.010 |
| stability_ari | 0.05 | 0.385 | 0.404 | -0.019 |
| granularity_score | 0.0 | 1.000 | 1.000 | 0.000 |
| robustness | 0.05 | 0.464 | 0.410 | 0.054 |
| silhouette | 0.01 | 0.069 | 0.069 | -0.000 |

First criterion (in priority order) where they differ beyond tolerance: **robustness** (winner 0.464 vs 0.410).
Within tolerance (effectively tied) on: semantic_cohesion_lift, stability_ari, granularity_score, silhouette.
Because the decision falls to a low-priority criterion, the margin is **small**; do not over-interpret it.

## Per-experiment candidate selection

**EXP_1_E5_KMEANS** -> `EXP_1_E5_KMEANS__k=100` (decided by tie_break(semantic_cohesion_lift, stability_ari))
- no non-pathological configuration - choosing among the least-flagged
- semantic_cohesion_lift: best=0.280 tol=0.02 -> kept EXP_1_E5_KMEANS__k=80, EXP_1_E5_KMEANS__k=100
- stability_ari: best=0.420 tol=0.05 -> kept EXP_1_E5_KMEANS__k=80, EXP_1_E5_KMEANS__k=100
- granularity_score: best=1.000 tol=0.0 -> kept EXP_1_E5_KMEANS__k=80, EXP_1_E5_KMEANS__k=100
- robustness: best=0.410 tol=0.05 -> kept EXP_1_E5_KMEANS__k=80, EXP_1_E5_KMEANS__k=100
- silhouette: best=0.071 tol=0.01 -> kept EXP_1_E5_KMEANS__k=80, EXP_1_E5_KMEANS__k=100
- still tied -> tie-break on raw cohesion lift then stability -> EXP_1_E5_KMEANS__k=100

**EXP_2_E5_HDBSCAN** -> `EXP_2_E5_HDBSCAN__min_cluster_size=30_min_samples=5` (decided by semantic_cohesion_lift)
- no non-pathological configuration - choosing among the least-flagged
- semantic_cohesion_lift: best=0.204 tol=0.02 -> kept EXP_2_E5_HDBSCAN__min_cluster_size=30_min_samples=5

**EXP_3_T5_KMEANS** -> `EXP_3_T5_KMEANS__k=100` (decided by tie_break(semantic_cohesion_lift, stability_ari))
- no non-pathological configuration - choosing among the least-flagged
- semantic_cohesion_lift: best=0.290 tol=0.02 -> kept EXP_3_T5_KMEANS__k=80, EXP_3_T5_KMEANS__k=100
- stability_ari: best=0.391 tol=0.05 -> kept EXP_3_T5_KMEANS__k=80, EXP_3_T5_KMEANS__k=100
- granularity_score: best=1.000 tol=0.0 -> kept EXP_3_T5_KMEANS__k=80, EXP_3_T5_KMEANS__k=100
- robustness: best=0.464 tol=0.05 -> kept EXP_3_T5_KMEANS__k=80, EXP_3_T5_KMEANS__k=100
- silhouette: best=0.069 tol=0.01 -> kept EXP_3_T5_KMEANS__k=80, EXP_3_T5_KMEANS__k=100
- still tied -> tie-break on raw cohesion lift then stability -> EXP_3_T5_KMEANS__k=100

**EXP_4_T5_HDBSCAN** -> `EXP_4_T5_HDBSCAN__min_cluster_size=20_min_samples=10` (decided by semantic_cohesion_lift)
- no non-pathological configuration - choosing among the least-flagged
- semantic_cohesion_lift: best=0.189 tol=0.02 -> kept EXP_4_T5_HDBSCAN__min_cluster_size=20_min_samples=10


In [7]:
# CELL 8 — optional result preview (reads the generated files; recalculates nothing)
import pandas as pd

cmp_path = out / "evaluation" / "experiment_comparison.csv"
if cmp_path.exists():
    cmp = pd.read_csv(cmp_path)
    display(cmp)
    def show(label, row):
        print(f"{label}: {row['config_id']}")
        print(f"  final leaf topics:      {row['n_clusters']}")
        print(f"  noise share (pre-rule): {row['noise_share']}")
        print(f"  semantic cohesion:      {row['semantic_cohesion']} (lift {row['semantic_cohesion_lift']})")
        print(f"  stability (subsample ARI): {row['stability_ari']} +/- {row['stability_ari_sd']}")
    for label, tag in [("WINNING PIPELINE", "1 (winner)"), ("SECOND-BEST PIPELINE", "2 (second)")]:
        hit = cmp[cmp["final_rank"].astype(str) == tag]
        if len(hit):
            show(label, hit.iloc[0])
else:
    print("experiment_comparison.csv not found")

mem_path = out / "final" / "task_topic_membership.csv"
top_path = out / "final" / "task_topics.csv"
if top_path.exists():
    print("\nfinal leaf topics in task_topics.csv:", len(pd.read_csv(top_path)))
if mem_path.exists():
    mem = pd.read_csv(mem_path)
    print("UNASSIGNED / noise statements in task_topic_membership.csv:", int((mem["topic_id"] == "UNASSIGNED").sum()))

,experiment_id,config_id,embedding_model,clustering_method,parameters,n_clusters,noise_share,semantic_cohesion,cohesion_random_baseline,semantic_cohesion_lift,...,cross_rep_ari,cross_rep_mean_best_jaccard,robustness,pathology_flags,nearest_neighbor_paraphrase_mrr,error_analysis_rows,semantic_cohesion_ci95,silhouette_ci95,noise_share_ci95,final_rank
0,EXP_1_E5_KMEANS,EXP_1_E5_KMEANS__k=100,intfloat/e5-base-v2,kmeans,k=100,100,0.000000,0.910788,0.876140,0.279736,...,0.244576,0.273892,0.410448,unstable,0.894726,190,"[0.909, 0.912]","[0.066, 0.071]",NaN,2 (second)
1,EXP_2_E5_HDBSCAN,EXP_2_E5_HDBSCAN__min_cluster_size=30_min_samp...,intfloat/e5-base-v2,hdbscan,"min_cluster_size=30,min_samples=5",4,0.938308,0.911978,0.889375,0.204316,...,0.000000,0.063479,-0.030043,too_few_clusters;dominant_cluster;excessive_noise,0.894726,180,"[0.909, 0.915]","[0.162, 0.185]","[0.924, 0.949]",NaN
2,EXP_3_T5_KMEANS,EXP_3_T5_KMEANS__k=100,sentence-transformers/sentence-t5-base,kmeans,k=100,100,0.000000,0.914050,0.878993,0.289711,...,0.244576,0.273892,0.464297,unstable,0.891496,190,"[0.913, 0.915]","[0.066, 0.071]",NaN,1 (winner)
3,EXP_4_T5_HDBSCAN,EXP_4_T5_HDBSCAN__min_cluster_size=20_min_samp...,sentence-transformers/sentence-t5-base,hdbscan,"min_cluster_size=20,min_samples=10",3,0.957877,0.925369,0.908014,0.188673,...,0.329862,0.186898,-0.069082,too_few_clusters;dominant_cluster;excessive_noise,0.891496,150,"[0.921, 0.930]","[0.146, 0.200]","[0.950, 0.965]",NaN


WINNING PIPELINE: EXP_3_T5_KMEANS__k=100
  final leaf topics:      100
  noise share (pre-rule): 0.0
  semantic cohesion:      0.9140501819835176 (lift 0.2897109824704051)
  stability (subsample ARI): 0.3845608561682331 +/- 0.0169405787472182
SECOND-BEST PIPELINE: EXP_1_E5_KMEANS__k=100
  final leaf topics:      100
  noise share (pre-rule): 0.0
  semantic cohesion:      0.9107881582791532 (lift 0.2797357278589375)
  stability (subsample ARI): 0.4039201144051098 +/- 0.0157998999249792

final leaf topics in task_topics.csv: 100
UNASSIGNED / noise statements in task_topic_membership.csv: 0
